In [1]:
import pandas as pd
import datasets
from datasets import Image as Image_ds # change name because of similar PIL module
from datasets import Dataset
import os
import requests 
from PIL import Image
from tqdm import tqdm

/Users/au672746/Library/CloudStorage/OneDrive-Aarhusuniversitet/CHC/art-multimodal-benchmark/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
omniart = pd.read_csv(os.path.join('data', 'omniart_v3_datadump.csv'))

In [12]:
omniart['image_url'].iloc[0]

'https://img00.deviantart.net/839b/i/2015/127/9/a/70_amx_by_xynphix-dyxa31.jpg'

In [3]:
omniart_paintings = omniart.query('general_type == "painting"')

In [14]:
omniart_paintings.columns

Index(['id', 'artwork_name', 'artist_full_name', 'artist_first_name',
       'artist_last_name', 'creation_year', 'century', 'source_url',
       'image_url', 'collection_origins', 'artwork_type', 'school',
       'original_id_in_collection', 'created_at', 'last_modified', 'omni_id',
       'created_by_id', 'general_type', 'geocoded', 'color_pallete',
       'dominant_color', 'palette_count'],
      dtype='object')

In [17]:
# save image thumbnail and fetch it using API
thumbnail = omniart_paintings['image_url'].iloc[0] # a 'thumbnail' is just a smaller version of the image (image native is around 10000x10000 pixels, takes forever to get)
img = Image.open(requests.get(thumbnail, stream = True).raw)

In [4]:
def download_image(url):
    '''
    Download image from thumbnail URL and encode it to HF feature
    '''
    feature = Image_ds(decode=False) 
    try:
        img = Image.open(requests.get(url, stream=True).raw) # stream=True enables to download image in chunks and not all at once

        # encode the PIL images to image feature
        image_encoded = feature.encode_example(img)

        return image_encoded

    except Exception as e:
        print(f"Error processing image: {e}")
        return pd.NA

In [5]:
omni_subset = omniart_paintings.iloc[:20].reset_index(drop=True)

In [6]:
images = [download_image(url) for url in tqdm(omni_subset['image_url'], desc= 'Downloading images')]

In [7]:
omni_subset['image'] = images

In [8]:
ds = Dataset.from_pandas(omni_subset).cast_column('image', Image_ds(decode=True)) # decode back to PIL image from raw bytes

# save HF dataset to data folder
ds.save_to_disk(os.path.join('data', 'omni_test'))

Saving the dataset (1/1 shards): 100%|██████████| 20/20 [00:00<00:00, 2009.44 examples/s]


In [9]:
import torch
from transformers import pipeline

In [10]:
import diffusers

In [11]:
pipe = pipeline(task="image-feature-extraction", model_name="stabilityai/stable-diffusion-3.5-large", pool=True)

No model was supplied, defaulted to google/vit-base-patch16-224 and revision 3f49326 (https://huggingface.co/google/vit-base-patch16-224).
Using a pipeline without specifying a model name and revision in production is not recommended.
Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
from diffusers import AutoencoderKL

# 1. Load the autoencoder model which will be used to decode the latents into image space. 
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae")

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


In [34]:
vae

AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
          )
        )
      )
      (1): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (c

In [37]:
import numpy as np
from torchvision import transforms
# Convert PIL image to tensor and add batch dimension
image = ds[0]['image']  # likely a PIL image

preprocess = transforms.Compose([
    transforms.ToTensor(),                     # Converts to (C, H, W) and normalizes to [0,1]
    transforms.Resize((512, 512))# Resize to match model input (optional, adjust as needed)
])

tensor = preprocess(image).unsqueeze(0)        # (1, C, H, W) — adds batch dimension

# Move to same device as the model
tensor = tensor.to(vae.device)

# Run through VAE
output = vae.encoder(tensor)

In [41]:
output.detach().numpy().shape

(1, 8, 64, 64)

In [49]:
pooling = torch.mean(output, dim=(2, 3))

In [52]:
pooling.detach().numpy().shape

(1, 8)

In [56]:
flattened = output.flatten(start_dim=1)

In [59]:
flattened.shape

torch.Size([1, 32768])

In [60]:
embedding = torch.mean(output, dim=(2, 3))

In [61]:
embedding.shape

torch.Size([1, 8])

In [64]:
omniart['artwork_type'].value_counts()

artwork_type
photo                                                                944425
image                                                                291507
painting                                                             223472
print                                                                 53618
emblem pictura                                                        32052
                                                                      ...  
armor parts-lance rests                                                   1
leatherwork|books and book bindings|manuscripts and illuminations         1
decorative arts                                                           1
firearms-cannon balls                                                     1
software                                                                  1
Name: count, Length: 519, dtype: int64

In [ ]:
omniart_paintings = omniart_paintings.loc[omniart_paintings['collection_origins'] != 'DeviantArt']

In [70]:
omniart_paintings.columns

Index(['id', 'artwork_name', 'artist_full_name', 'artist_first_name',
       'artist_last_name', 'creation_year', 'century', 'source_url',
       'image_url', 'collection_origins', 'artwork_type', 'school',
       'original_id_in_collection', 'created_at', 'last_modified', 'omni_id',
       'created_by_id', 'general_type', 'geocoded', 'color_pallete',
       'dominant_color', 'palette_count'],
      dtype='object')

In [72]:
omniart_paintings.loc[omniart_paintings['collection_origins'] == 'Rijksmuseum']

,id,artwork_name,artist_full_name,artist_first_name,artist_last_name,creation_year,century,source_url,image_url,collection_origins,...,original_id_in_collection,created_at,last_modified,omni_id,created_by_id,general_type,geocoded,color_pallete,dominant_color,palette_count


In [76]:
omniart_paintings['general_type'].value_counts()

general_type
painting    150112
Name: count, dtype: int64

In [81]:
omniart['collection_origins'].value_counts()

collection_origins
The British Library [Flickr]    942883
Brill Iconclass Arkyves         376527
The Met 17                      205702
DeviantArt                      184167
WikiArts 17                     130436
MOMA - New York                  62259
Web Gallery of Art '17           44299
Name: count, dtype: int64

In [86]:
omniart.columns

Index(['id', 'artwork_name', 'artist_full_name', 'artist_first_name',
       'artist_last_name', 'creation_year', 'century', 'source_url',
       'image_url', 'collection_origins', 'artwork_type', 'school',
       'original_id_in_collection', 'created_at', 'last_modified', 'omni_id',
       'created_by_id', 'general_type', 'geocoded', 'color_pallete',
       'dominant_color', 'palette_count'],
      dtype='object')

In [109]:
omniart['palette_count']

0          [4009, 7637, 3303, 6147, 4742, 3908, 7216, 514...
1          [3686, 5229, 6630, 5395, 4857, 6420, 5178, 468...
2          [6226, 6056, 4293, 6101, 5185, 5292, 6392, 384...
3          [1007, 4537, 4708, 4466, 4432, 2946, 4776, 448...
4          [4225, 4308, 4914, 6274, 6911, 6261, 6753, 421...
                                 ...                        
1946268    [5783, 3494, 4995, 8378, 4264, 6912, 7478, 352...
1946269    [2858, 6217, 5429, 6842, 5640, 7505, 9864, 532...
1946270    [3761, 5448, 4982, 6739, 3506, 4867, 9716, 668...
1946271    [3493, 3989, 4828, 4144, 6692, 4971, 4292, 478...
1946272    [3673, 4253, 5392, 6641, 3191, 7046, 4680, 706...
Name: palette_count, Length: 1946273, dtype: object

In [83]:
omni_test = pd.read_csv(os.path.join('data', 'omniart_tester.csv'))

In [84]:
omni_test.columns

Index(['id', 'artwork_name', 'artist_full_name', 'artist_first_name',
       'artist_last_name', 'creation_year', 'century', 'source_url',
       'image_url', 'collection_origins', 'artwork_type', 'school',
       'original_id_in_collection', 'created_at', 'last_modified', 'omni_id',
       'created_by_id', 'general_type', 'geocoded', 'color_pallete',
       'dominant_color', 'palette_count'],
      dtype='object')

In [110]:
import timm

In [113]:
timm.list_models(pretrained=True)

['bat_resnext26ts.ch_in1k',
 'beit_base_patch16_224.in22k_ft_in22k',
 'beit_base_patch16_224.in22k_ft_in22k_in1k',
 'beit_base_patch16_384.in22k_ft_in22k_in1k',
 'beit_large_patch16_224.in22k_ft_in22k',
 'beit_large_patch16_224.in22k_ft_in22k_in1k',
 'beit_large_patch16_384.in22k_ft_in22k_in1k',
 'beit_large_patch16_512.in22k_ft_in22k_in1k',
 'beitv2_base_patch16_224.in1k_ft_in1k',
 'beitv2_base_patch16_224.in1k_ft_in22k',
 'beitv2_base_patch16_224.in1k_ft_in22k_in1k',
 'beitv2_large_patch16_224.in1k_ft_in1k',
 'beitv2_large_patch16_224.in1k_ft_in22k',
 'beitv2_large_patch16_224.in1k_ft_in22k_in1k',
 'botnet26t_256.c1_in1k',
 'caformer_b36.sail_in1k',
 'caformer_b36.sail_in1k_384',
 'caformer_b36.sail_in22k',
 'caformer_b36.sail_in22k_ft_in1k',
 'caformer_b36.sail_in22k_ft_in1k_384',
 'caformer_m36.sail_in1k',
 'caformer_m36.sail_in1k_384',
 'caformer_m36.sail_in22k',
 'caformer_m36.sail_in22k_ft_in1k',
 'caformer_m36.sail_in22k_ft_in1k_384',
 'caformer_s18.sail_in1k',
 'caformer_s18.s

In [116]:
timm.list_models('*clip*')

['eva02_base_patch16_clip_224',
 'eva02_enormous_patch14_clip_224',
 'eva02_large_patch14_clip_224',
 'eva02_large_patch14_clip_336',
 'eva_giant_patch14_clip_224',
 'vit_base_patch16_clip_224',
 'vit_base_patch16_clip_384',
 'vit_base_patch16_clip_quickgelu_224',
 'vit_base_patch32_clip_224',
 'vit_base_patch32_clip_256',
 'vit_base_patch32_clip_384',
 'vit_base_patch32_clip_448',
 'vit_base_patch32_clip_quickgelu_224',
 'vit_betwixt_patch32_clip_224',
 'vit_giant_patch14_clip_224',
 'vit_gigantic_patch14_clip_224',
 'vit_huge_patch14_clip_224',
 'vit_huge_patch14_clip_336',
 'vit_huge_patch14_clip_378',
 'vit_huge_patch14_clip_quickgelu_224',
 'vit_huge_patch14_clip_quickgelu_378',
 'vit_large_patch14_clip_224',
 'vit_large_patch14_clip_336',
 'vit_large_patch14_clip_quickgelu_224',
 'vit_large_patch14_clip_quickgelu_336',
 'vit_medium_patch16_clip_224',
 'vit_medium_patch32_clip_224',
 'vit_xsmall_patch16_clip_224']

In [117]:
import torch
from PIL import Image
from transformers import AutoModel, CLIPImageProcessor

In [119]:
model = AutoModel.from_pretrained(
    'OpenGVLab/InternViT-6B-448px-V2_5',
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True).cuda().eval()

Encountered exception while importing flash_attn: No module named 'flash_attn'


ImportError: This modeling file requires the following packages that were not found in your environment: flash_attn. Run `pip install flash_attn`